In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .config("master", "yarn") \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.adaptive.coalescePartitions.enabled", True) \
    .config("spark.sql.autoBroadcastJoinThreshold", -1) \
    .config("spark.sql.sources.bucketing.enabled", True) \
    .config("spark.executor.memory", "512M") \
    .config("spark.executor.memoryOverhead", "384M") \
    .config("spark.driver.memory", "512M") \
    .config("spark.driver.memoryOverhead", "384M") \
    .appName("hw2-student-14-churn-simple") \
    .getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/jupyterhub/lib/python3.9/site-packages/pyspark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-log4j12-1.7.25.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2025-08-09 15:54:58 WARN  spark.util.Utils:73 - Service 'sparkDriver' could not bind on port 40000. Attempting port 40001.
2025-08-09 15:54:58 WARN  spark.util.Utils:73 - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2025-08-09 15:54:59 WARN  deploy.yarn.Client:73 - Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
2025-08-09 15:55:05 WARN  spark.util.Utils:73 - Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 40500. Attempting port 40501.


- Параметры по памяти определить самостоятельно
- Включение AQE или Auto Broadcast => не зачет

# Задание 1

## Входные данные
- Таблица `hw.telecom_churn` - с данными по оттоку телеком оператора в США
- Справочник `hw.states` с названиями штатов
- Справочник `hw.population` с численностью населения территорий (определяется полем `area code`) внутри штатов
- Террия с численностью населения меньше `10_000` считается **мелкой**

## Что нужно сделать
1. Посчитать количество отточных и неотточных абонентов (поле `churn`), исключив **мелкие** территории
2. Отчет должен быть выполнен в разрезе **каждого штата** с его полным наименованием
3. Описать возникающие узкие места при выполнении данной операции
4. Применить один из способов оптимизации для ускорения выполнения запроса (при допущении, что справочник численности населения **сильно меньше** основных данных)
5. Если существует еще какой-то способ, применить также и его отдельно от п.4 (при допущении, что справочник численности населения **сопоставим по размеру** с основными данными)
6. Кратко описать реализованные способы и в чем их практическая польза

- P.S. Одним из выбранных способов должен быть `Bucket specific join`
- P.P.S. При обосновании предлагаем прикладывать запуска команды `df.explain()`

In [3]:
churn_df = spark.table("hw.telecom_churn")
states_df = spark.table("hw.states")
population_df = spark.table("hw.population")

2025-08-09 15:55:12 INFO  hive.conf.HiveConf:187 - Found configuration file file:/etc/hive/conf/hive-site.xml
2025-08-09 15:55:12 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.metastore.warehouse.external.dir does not exist
2025-08-09 15:55:12 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.enforce.bucketing does not exist
2025-08-09 15:55:12 WARN  hive.client.HiveClientImpl:73 - Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
2025-08-09 15:55:12 INFO  hive.metastore:405 - Trying to connect to metastore with URI thrift://dataops-hadoop-mn-1:9083
2025-08-09 15:55:12 INFO  hive.metastore:479 - Opened a connection to metastore, current connections: 1
2025-08-09 15:55:12 INFO  hive.metastore:532 - Connected to metastore.


In [4]:
churn_df.show(1, False, True)

[Stage 0:>                                                          (0 + 1) / 1]

-RECORD 0--------------------------
 id                     | 280      
 state                  | CA       
 account length         | 75       
 area code              | 339      
 phone number           | 341-1916 
 international plan     | no       
 voice mail plan        | yes      
 number vmail messages  | 19       
 total day minutes      | 210.3    
 total day calls        | 90       
 total day charge       | 35.75    
 total eve minutes      | 241.8    
 total eve calls        | 87       
 total eve charge       | 20.55    
 total night minutes    | 215.7    
 total night calls      | 102      
 total night charge     | 9.71     
 total intl minutes     | 13.1     
 total intl calls       | 3        
 total intl charge      | 3.54     
 customer service calls | 4        
 churn                  | False    
 val                    | 339      
only showing top 1 row



In [5]:
states_df.show(1, False, True)

-RECORD 0--------------
 state_id   | MD       
 state_name | Maryland 
only showing top 1 row



In [6]:
population_df.show(1, False, True)

-RECORD 0-----------
 area code  | 148   
 population | 74981 
only showing top 1 row



### Решение

In [7]:
df = churn_df.join(
    population_df.filter('population >= 10000'),
    on='area code'
)

In [8]:
result = df.join(
    states_df,
    df["state"] == states_df["state_id"]
).groupBy("state_name").pivot("churn").count()


In [9]:
result.explain()
result.show(truncate=False)

== Physical Plan ==
*(11) Project [state_name#47, __pivot_count(1) AS count AS `count(1) AS count`#281[0] AS False#282L, __pivot_count(1) AS count AS `count(1) AS count`#281[1] AS True#283L]
+- HashAggregate(keys=[state_name#47], functions=[pivotfirst(churn#21, count(1) AS count#275L, False, True, 0, 0)])
   +- Exchange hashpartitioning(state_name#47, 200), ENSURE_REQUIREMENTS, [plan_id=436]
      +- HashAggregate(keys=[state_name#47], functions=[partial_pivotfirst(churn#21, count(1) AS count#275L, False, True, 0, 0)])
         +- *(10) HashAggregate(keys=[state_name#47, churn#21], functions=[count(1)])
            +- Exchange hashpartitioning(state_name#47, churn#21, 200), ENSURE_REQUIREMENTS, [plan_id=431]
               +- *(9) HashAggregate(keys=[state_name#47, churn#21], functions=[partial_count(1)])
                  +- *(9) Project [churn#21, state_name#47]
                     +- *(9) SortMergeJoin [state#1], [state_id#46], Inner
                        :- *(6) Sort [state#1 AS

+--------------------+-----+----+
|state_name          |False|True|
+--------------------+-----+----+
|Utah                |62   |10  |
|Hawaii              |42   |3   |
|Minnesota           |69   |15  |
|Ohio                |68   |10  |
|Oregon              |67   |11  |
|Arkansas            |35   |9   |
|Texas               |54   |18  |
|North Dakota        |56   |6   |
|Pennsylvania        |37   |8   |
|Connecticut         |56   |8   |
|Vermont             |65   |8   |
|Nebraska            |56   |5   |
|Nevada              |52   |14  |
|Washington          |52   |14  |
|Illinois            |49   |5   |
|Oklahoma            |52   |9   |
|District of Columbia|43   |4   |
|Delaware            |48   |8   |
|Alaska              |44   |3   |
|New Mexico          |56   |6   |
+--------------------+-----+----+
only showing top 20 rows



http://dataops-hadoop-mn-2:18080/history/application_1752758811171_3249/jobs/

Двойной join - сначала с population, потом с states
Pivot операция - требует перегруппировки данных

### Оптимизация 1

In [10]:
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .config("master", "yarn") \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.adaptive.coalescePartitions.enabled", True) \
    .config("spark.sql.autoBroadcastJoinThreshold", -1) \
    .config("spark.sql.sources.bucketing.enabled", True) \
    .config("spark.executor.memory", "512M") \
    .config("spark.executor.memoryOverhead", "384M") \
    .config("spark.driver.memory", "512M") \
    .config("spark.driver.memoryOverhead", "384M") \
    .appName("hw2-student-14-churn-opt1") \
    .getOrCreate()

2025-08-09 15:55:52 WARN  spark.util.Utils:73 - Service 'sparkDriver' could not bind on port 40000. Attempting port 40001.
2025-08-09 15:55:52 WARN  spark.util.Utils:73 - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2025-08-09 15:55:52 WARN  deploy.yarn.Client:73 - Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
2025-08-09 15:55:58 WARN  spark.util.Utils:73 - Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 40500. Attempting port 40501.
2025-08-09 15:55:58 WARN  scheduler.cluster.YarnSchedulerBackend$YarnSchedulerEndpoint:73 - Attempted to request executors before the AM has registered!


In [11]:
churn_df = spark.table("hw.telecom_churn")
states_df = spark.table("hw.states")
population_df = spark.table("hw.population")

2025-08-09 15:56:03 INFO  hive.conf.HiveConf:187 - Found configuration file file:/etc/hive/conf/hive-site.xml
2025-08-09 15:56:03 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.metastore.warehouse.external.dir does not exist
2025-08-09 15:56:03 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.enforce.bucketing does not exist
2025-08-09 15:56:03 WARN  hive.client.HiveClientImpl:73 - Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
2025-08-09 15:56:03 INFO  hive.metastore:405 - Trying to connect to metastore with URI thrift://dataops-hadoop-mn-1:9083
2025-08-09 15:56:03 INFO  hive.metastore:479 - Opened a connection to metastore, current connections: 1
2025-08-09 15:56:03 INFO  hive.metastore:532 - Connected to metastore.


In [15]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS student_user14")

churn_for_bucketing = churn_df.repartition(1) \
    .withColumn("area code", F.col("area code").cast("bigint"))

population_for_bucketing = population_df.repartition(1) \
    .withColumn("area code", F.col("area code").cast("bigint"))

churn_for_bucketing.write \
    .mode("overwrite") \
    .bucketBy(8, "area code") \
    .saveAsTable(f"student_user14.telecom_churn_bucketed")

population_for_bucketing.write \
    .mode("overwrite") \
    .bucketBy(8, "area code") \
    .saveAsTable(f"student_user14.population_bucketed")

2025-08-09 16:03:30 INFO  plugin.sqlstd.SQLStdHiveAccessController:95 - Created SQLStdHiveAccessController for session context : HiveAuthzSessionContext [sessionString=57a0bd7b-98a7-4463-b2ce-e140955fbbac, clientType=HIVECLI]
2025-08-09 16:03:30 WARN  ql.session.SessionState:907 - METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2025-08-09 16:03:30 INFO  hive.metastore:313 - Mestastore configuration hive.metastore.filter.hook changed from org.apache.hadoop.hive.metastore.DefaultMetaStoreFilterHookImpl to org.apache.hadoop.hive.ql.security.authorization.plugin.AuthorizationMetaStoreFilterHook
2025-08-09 16:03:30 INFO  hive.metastore:562 - Closed a connection to metastore, current connections: 0
2025-08-09 16:03:30 INFO  hive.metastore:405 - Trying to connect to metastore with URI thrift://dataops-hadoop-mn-1:9083
2025-08-09 16:03:30 INFO  hive.metastore:479 - Opened a connection to metastore, current connection

In [16]:
# spark.catalog.listDatabases()
! hdfs dfs -ls hdfs://demo/warehouse/tablespace/managed/hive/student_user14.db/

Found 2 items
drwxr-x---   - student_user14 hive          0 2025-08-09 16:03 hdfs://demo/warehouse/tablespace/managed/hive/student_user14.db/population_bucketed
drwxr-x---   - student_user14 hive          0 2025-08-09 16:03 hdfs://demo/warehouse/tablespace/managed/hive/student_user14.db/telecom_churn_bucketed


In [18]:
churn_bucketed = spark.table(f"student_user14.telecom_churn_bucketed") \
    .withColumn("churn", F.col("churn") == "True")
population_bucketed = spark.table(f"student_user14.population_bucketed")

In [20]:
filtered_churn = churn_bucketed.join(population_bucketed, on="area code") \
    .filter(F.col("population") >= 10_000)

result = filtered_churn.join(states_df.hint("broadcast"), on=F.col("state") == F.col("state_id")) \
    .groupBy("state_name") \
    .agg(
        F.sum(F.when(F.col("churn") == True, 1).otherwise(0)).alias("churn_true"),
        F.sum(F.when(F.col("churn") == False, 1).otherwise(0)).alias("churn_false")
    )

In [21]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

In [23]:
result.explain()

== Physical Plan ==
*(5) HashAggregate(keys=[state_name#369], functions=[sum(CASE WHEN churn#712 THEN 1 ELSE 0 END), sum(CASE WHEN NOT churn#712 THEN 1 ELSE 0 END)])
+- Exchange hashpartitioning(state_name#369, 8), ENSURE_REQUIREMENTS, [plan_id=1626]
   +- *(4) HashAggregate(keys=[state_name#369], functions=[partial_sum(CASE WHEN churn#712 THEN 1 ELSE 0 END), partial_sum(CASE WHEN NOT churn#712 THEN 1 ELSE 0 END)])
      +- *(4) Project [churn#712, state_name#369]
         +- *(4) BroadcastHashJoin [state#615], [state_id#368], Inner, BuildRight, false
            :- *(4) Project [state#615, churn#712]
            :  +- *(4) SortMergeJoin [area code#617L], [area code#685L], Inner
            :     :- *(1) Sort [area code#617L ASC NULLS FIRST], false, 0
            :     :  +- *(1) Project [state#615, area code#617L, (churn#635 = True) AS churn#712]
            :     :     +- *(1) Filter (isnotnull(area code#617L) AND isnotnull(state#615))
            :     :        +- *(1) ColumnarToRow

In [24]:
result.show(truncate=False)

[Stage 29:===========================================>              (6 + 2) / 8]

+--------------+----------+-----------+
|state_name    |churn_true|churn_false|
+--------------+----------+-----------+
|Oklahoma      |9         |52         |
|Maryland      |17        |53         |
|Nevada        |14        |52         |
|Utah          |10        |62         |
|Alabama       |7         |58         |
|Hawaii        |3         |42         |
|Montana       |14        |54         |
|Massachusetts |11        |54         |
|Arizona       |4         |53         |
|Florida       |8         |45         |
|Indiana       |9         |59         |
|Ohio          |10        |68         |
|Virginia      |5         |72         |
|New Mexico    |6         |56         |
|Rhode Island  |6         |59         |
|Michigan      |16        |57         |
|North Dakota  |6         |56         |
|Louisiana     |4         |47         |
|North Carolina|11        |57         |
|Vermont       |8         |65         |
+--------------+----------+-----------+
only showing top 20 rows



Обе таблицы записаны в бакет по area_code. 
К маленькой таблице 'states_df' применяем broadcast, получаем BroadcastHashJoin эффективный вид джоина для малых данных.

http://dataops-hadoop-mn-2:18080/history/application_1752758811171_3253/jobs/

### Оптимизация 2

# Задание 2

## Входные данные

Таблица `hw.transactions` - информация о длительности просмотра контента пользователями:
1. user_uid — уникальный идентификатор пользователя
2. element_uid — уникальный идентификатор контента
3. watched_time — время просмотра в секундах

Справочник `hw.catalogue` - каталог с описанием контента и метаинформации по нему:
1. type — тип элемента
2. duration — длительность в минутах (средняя длительность эпизода в случае с сериалами и многосерийными фильмами), округлённая до десятков
3. attributes — анонимизированные атрибуты данного элемента
4. availability — доступные права на элемент(subscription, purchase, rent)
5. feature_1 — анонимизированная вещественная переменная
6. feature_2 — анонимизированная вещественная переменная
7. feature_3 — анонимизированная порядковая переменная
8. feature_4 — анонимизированная вещественная переменная
9. feature_5 — анонимизированная вещественная переменная

## Что нужно сделать
1. Выполните join основных данных со справочником используя DataFrame API (по колонке id для контента - `element_uid`)
2. Описать проблему в датасетах с точки зрения обработки Spark
3. Решить задачу любым способом
4. Решить задачу с помощью salt-join подхода

P.S. Как вы можете заметить при просмотре данных по пользователями, нужный нам ключ для операции будет перекошен (90% строк представлены на фильм, очень популярный среди смотревших) - это нужно доказать в рамках п.2

### Решение

In [35]:
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .config("master", "yarn") \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.autoBroadcastJoinThreshold", -1) \
    .config("spark.sql.sources.bucketing.enabled", True) \
    .config("spark.executor.memory", "650M") \
    .config("spark.driver.memory", "650M") \
    .appName("hw2-student-14-trans") \
    .getOrCreate()

2025-08-09 18:31:06 WARN  spark.util.Utils:73 - Service 'sparkDriver' could not bind on port 40000. Attempting port 40001.
2025-08-09 18:31:06 WARN  spark.util.Utils:73 - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2025-08-09 18:31:06 WARN  deploy.yarn.Client:73 - Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
2025-08-09 18:31:12 WARN  spark.util.Utils:73 - Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 40500. Attempting port 40501.


In [36]:
transactions_df = spark.table("hw.transactions")
catalogue_df = spark.table("hw.catalogue")

2025-08-09 18:31:16 INFO  hive.conf.HiveConf:187 - Found configuration file file:/etc/hive/conf/hive-site.xml
2025-08-09 18:31:16 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.metastore.warehouse.external.dir does not exist
2025-08-09 18:31:16 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.enforce.bucketing does not exist
2025-08-09 18:31:16 WARN  hive.client.HiveClientImpl:73 - Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
2025-08-09 18:31:16 INFO  hive.metastore:405 - Trying to connect to metastore with URI thrift://dataops-hadoop-mn-1:9083
2025-08-09 18:31:16 INFO  hive.metastore:479 - Opened a connection to metastore, current connections: 1
2025-08-09 18:31:16 INFO  hive.metastore:532 - Connected to metastore.


In [37]:
print("transactions_df count:", transactions_df.count())
print("catalogue_df count:", catalogue_df.count())

transactions_df count: 30070692


[Stage 2:===============================================>       (173 + 0) / 200]

catalogue_df count: 10200


In [38]:
catalogue_df.show(1,False,True)

-RECORD 0----------------------------------------------------------------------
 type         | movie                                                          
 availability | [purchase]                                                     
 duration     | 90                                                             
 feature_1    | 1.2780953E7                                                    
 feature_2    | 0.7544667                                                      
 feature_3    | 6                                                              
 feature_4    | 0.9461333                                                      
 feature_5    | 0.0                                                            
 attributes   | [17545, 17546, 396, 17547, 3771, 124, 910, 17548, 15546, 3277] 
 id           | 6779                                                           
only showing top 1 row



In [39]:
catalogue_df_ = catalogue_df.withColumnRenamed("id", "element_uid")

result = transactions_df \
    .join(
        catalogue_df_,
        on="element_uid", how="inner"
    )

In [40]:
result.explain()

== Physical Plan ==
*(5) Project [element_uid#1153, user_uid#1152, watched_time#1154, type#1158, availability#1159, duration#1160, feature_1#1161, feature_2#1162, feature_3#1163, feature_4#1164, feature_5#1165, attributes#1166]
+- *(5) SortMergeJoin [element_uid#1153], [element_uid#1243], Inner
   :- *(2) Sort [element_uid#1153 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(element_uid#1153, 200), ENSURE_REQUIREMENTS, [plan_id=2266]
   :     +- *(1) Filter isnotnull(element_uid#1153)
   :        +- *(1) ColumnarToRow
   :           +- FileScan orc hw.transactions[user_uid#1152,element_uid#1153,watched_time#1154] Batched: true, DataFilters: [isnotnull(element_uid#1153)], Format: ORC, Location: InMemoryFileIndex(1 paths)[hdfs://demo/warehouse/tablespace/managed/hive/hw.db/transactions], PartitionFilters: [], PushedFilters: [IsNotNull(element_uid)], ReadSchema: struct<user_uid:string,element_uid:string,watched_time:string>
   +- *(4) Sort [element_uid#1243 ASC NULLS FIRST],

In [41]:
result.show(2, False, True)

[Stage 6:========================================================>(69 + 1) / 70]

-RECORD 0---------------------------------------------------------------
 element_uid  | 10096                                                   
 user_uid     | 86501                                                   
 watched_time | 6132                                                    
 type         | movie                                                   
 availability | [purchase, rent]                                        
 duration     | 90                                                      
 feature_1    | 1.6498549E7                                             
 feature_2    | 0.72082573                                              
 feature_3    | 2                                                       
 feature_4    | 1.104374                                                
 feature_5    | 0.6547074                                               
 attributes   | [13904, 27089, 3815, 7, 27090, 32, 308, 170, 20, 27091] 
-RECORD 1------------------------------------------

http://dataops-hadoop-mn-2:18080/history/application_1752758811171_3286/jobs/

Присутствует ресурсоемкий SortMergeJoin, 2 шафла.
Некоторые element_uid встречаются очень часто (популярный контент), что приводит к неравномерному распределению данных между экзекьюторами

### Решение с оптимизацией

In [49]:
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .config("master", "yarn") \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.autoBroadcastJoinThreshold", -1) \
    .config("spark.sql.sources.bucketing.enabled", True) \
    .config("spark.executor.memory", "650M") \
    .config("spark.driver.memory", "650M") \
    .appName("hw2-student-14-trans-opt") \
    .getOrCreate()

2025-08-09 18:47:50 WARN  spark.util.Utils:73 - Service 'sparkDriver' could not bind on port 40000. Attempting port 40001.
2025-08-09 18:47:50 WARN  spark.util.Utils:73 - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2025-08-09 18:47:50 WARN  deploy.yarn.Client:73 - Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
2025-08-09 18:47:55 WARN  spark.util.Utils:73 - Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 40500. Attempting port 40501.
2025-08-09 18:47:55 WARN  scheduler.cluster.YarnSchedulerBackend$YarnSchedulerEndpoint:73 - Attempted to request executors before the AM has registered!


In [50]:
transactions_df = spark.table("hw.transactions")
catalogue_df = spark.table("hw.catalogue").withColumnRenamed("id", "element_uid")

2025-08-09 18:47:59 INFO  hive.conf.HiveConf:187 - Found configuration file file:/etc/hive/conf/hive-site.xml
2025-08-09 18:48:00 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.metastore.warehouse.external.dir does not exist
2025-08-09 18:48:00 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.enforce.bucketing does not exist
2025-08-09 18:48:00 WARN  hive.client.HiveClientImpl:73 - Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
2025-08-09 18:48:00 INFO  hive.metastore:405 - Trying to connect to metastore with URI thrift://dataops-hadoop-mn-1:9083
2025-08-09 18:48:00 INFO  hive.metastore:479 - Opened a connection to metastore, current connections: 1
2025-08-09 18:48:00 INFO  hive.metastore:532 - Connected to metastore.


In [51]:
from pyspark.sql.functions import concat_ws, col, expr, floor, rand, crc32
num_salts = 9

catalogue_salted = catalogue_df.withColumn(
    "salt",
    (crc32(col("element_uid")) % num_salts).cast("int")
).withColumn(
    "salted_element_uid",
    concat_ws("_", col("element_uid"), col("salt"))
)

transactions_salted = transactions_df.withColumn(
    "salt", (crc32(col("element_uid")) % num_salts).cast("int")
).withColumn(
    "salted_element_uid",
    concat_ws("_", col("element_uid"), col("salt"))
).withColumnRenamed("element_uid", "element_uid_salt")

result_salted = transactions_salted.join(
    broadcast(catalogue_salted),
    on="salted_element_uid",
    how="inner"
)

all_columns = result_salted.columns
columns_to_keep = [c for c in all_columns if "salt" not in c.lower()]
print("Columns to keep:", columns_to_keep)
final_df = result_salted.select(*columns_to_keep)

Columns to keep: ['user_uid', 'watched_time', 'type', 'availability', 'duration', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'attributes', 'element_uid']


In [52]:
final_df.explain()

== Physical Plan ==
*(2) Project [user_uid#1523, watched_time#1525, type#1529, availability#1530, duration#1531, feature_1#1532, feature_2#1533, feature_3#1534, feature_4#1535, feature_5#1536, attributes#1537, element_uid#1549]
+- *(2) BroadcastHashJoin [salted_element_uid#1592], [salted_element_uid#1573], Inner, BuildRight, false
   :- *(2) Project [user_uid#1523, watched_time#1525, concat_ws(_, element_uid#1524, cast(cast((crc32(cast(element_uid#1524 as binary)) % 9) as int) as string)) AS salted_element_uid#1592]
   :  +- *(2) ColumnarToRow
   :     +- FileScan orc hw.transactions[user_uid#1523,element_uid#1524,watched_time#1525] Batched: true, DataFilters: [], Format: ORC, Location: InMemoryFileIndex(1 paths)[hdfs://demo/warehouse/tablespace/managed/hive/hw.db/transactions], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<user_uid:string,element_uid:string,watched_time:string>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[10, string, false]),false), [pl

In [53]:
final_df.show(2, False, True)

-RECORD 0---------------------------------------------------------------------------------------------
 user_uid     | 50451                                                                                 
 watched_time | 7593                                                                                  
 type         | movie                                                                                 
 availability | [purchase, rent, subscription]                                                        
 duration     | 130                                                                                   
 feature_1    | 4.2346732E7                                                                           
 feature_2    | 0.74050003                                                                            
 feature_3    | 45                                                                                    
 feature_4    | 1.1419294                                                

In [ ]:
http://dataops-hadoop-mn-2:18080/history/application_1752758811171_3295/jobs/

In [ ]:
Добавляю соль к ключам соединения (element_uid), чтобы распределить популярный контент по нескольким экзекьюторам.
Использую Broadcast Join для catalogue_salted, так как это данные маленького объема.
Как итог - Равномерная нагрузка между экзекюторами, отсутствие шаффлов.

# Задание 3

## Входные данные

Таблица `hw.transactions`  — информация о длительности просомтра контента пользователями:

1. user_uid — уникальный идентификатор пользователя
2. element_uid — уникальный идентификатор контента
3. watched_time — время просмотра в секундах

Таблица `hw.ratings`  — информация об оценках, поставленных пользователями:

1. user_uid — уникальный идентификатор пользователя
2. element_uid — уникальный идентификатор контента
3. rating — поставленный пользователем рейтинг

Справочник `hw.user_uids`  — выборка пользователей:
1. user_uid — уникальный идентификатор пользователя


## Что нужно сделать
Для каждого пользователя из выборки посчитать:
1. Максимальное и минимальное время просмотра фильмов с оценками 8, 9 и 10
2. Название фичи должно быть в формате `feat_<агрегирующая_функция>_watched_time_rating_<оценка>`
3. Если у пользователь не ставил оценки 8, 9 и 10 то значение фичей должно быть null
4. Описать принятые при разработки кода решения и возможные оптимизации

P.S. На каждом этапе обработки должно быть должны агрегироваться минимально возможные объемы данных (сокращаем затраты на shuflle)

### Решение

In [2]:
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .config("master", "yarn") \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.autoBroadcastJoinThreshold", -1) \
    .config("spark.sql.sources.bucketing.enabled", True) \
    .config("spark.executor.memory", "650M") \
    .config("spark.executor.memoryOverhead", "384M") \
    .config("spark.driver.memory", "650M") \
    .config("spark.driver.memoryOverhead", "384M") \
    .appName("hw2-student-14-movies") \
    .getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/jupyterhub/lib/python3.9/site-packages/pyspark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-log4j12-1.7.25.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2025-08-09 19:47:45 WARN  spark.util.Utils:73 - Service 'sparkDriver' could not bind on port 40000. Attempting port 40001.
2025-08-09 19:47:45 WARN  spark.util.Utils:73 - Service 'sparkDriver' could not bind on port 40001. Attempting port 40002.
2025-08-09 19:47:45 WARN  spark.util.Utils:73 - Service 'sparkDriver' could not bind on port 40002. Attempting port 40003.
2025-08-09 19:47:45 WARN  spark.util.Utils:73 - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2025-08-09 19:47:46 WARN  deploy.yarn.Client:73 - Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
2025-08-09 19:47:52 WARN  spark.util.Utils:73 - Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 40500. Attempting port 40501.
2025-08-09 19:47:52 WARN  scheduler.cluster.YarnSchedulerBackend$YarnSchedulerEndpoint:73 - Attempted to request executors before the AM has registered!


In [3]:
transactions_df = spark.table("hw.transactions")
ratings_df = spark.table("hw.ratings")
user_uids_df = spark.table("hw.user_uids")

2025-08-09 19:47:59 INFO  hive.conf.HiveConf:187 - Found configuration file file:/etc/hive/conf/hive-site.xml
2025-08-09 19:47:59 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.metastore.warehouse.external.dir does not exist
2025-08-09 19:47:59 WARN  hive.conf.HiveConf:4122 - HiveConf of name hive.enforce.bucketing does not exist
2025-08-09 19:47:59 WARN  hive.client.HiveClientImpl:73 - Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
2025-08-09 19:47:59 INFO  hive.metastore:405 - Trying to connect to metastore with URI thrift://dataops-hadoop-mn-1:9083
2025-08-09 19:47:59 INFO  hive.metastore:479 - Opened a connection to metastore, current connections: 1
2025-08-09 19:47:59 INFO  hive.metastore:532 - Connected to metastore.


In [5]:
transactions_df.show(1,False,True)

-RECORD 0-------------
 user_uid     | 50451 
 element_uid  | 2714  
 watched_time | 7593  
only showing top 1 row



In [6]:
ratings_df.show(1,False,True)

-RECORD 0-------------
 user_uid    | 230825 
 element_uid | 5551   
 rating      | 10     
only showing top 1 row



In [7]:
user_uids_df.show(1,False,True)

-RECORD 0----------
 user_uid | 110138 
only showing top 1 row



In [8]:
duplicates_keys = transactions_df \
    .groupBy("user_uid", "element_uid") \
    .count() \
    .filter(F.col("count") > 1) \
    .select("user_uid", "element_uid") 

duplicates_full = transactions_df.join(
    duplicates_keys,
    on=["user_uid", "element_uid"],
    how="inner"
)

duplicates_full.orderBy("user_uid", "element_uid").show()

[Stage 6:======================================================>(199 + 1) / 200]

+--------+-----------+------------+
|user_uid|element_uid|watched_time|
+--------+-----------+------------+
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
|  100001|       2714|          70|
+--------+-----------+------------+
only showing top 20 rows



In [9]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

In [10]:
filtered_ratings = ratings_df \
    .filter(F.col("rating").isin([8, 9, 10])) \
    .join(user_uids_df, on="user_uid", how="inner")

In [12]:
from pyspark.sql.functions import broadcast

In [21]:
transactions_clean = transactions_df.dropDuplicates(["user_uid", "element_uid", "watched_time"])

filtered_ratings = ratings_df \
    .filter(F.col("rating").isin([8, 9, 10])) \
    .join(broadcast(user_uids_df), on="user_uid", how="left")

ratings_with_time = transactions_clean.join(
    broadcast(filtered_ratings),
    on=["user_uid", "element_uid"],
    how="inner"
)

agg_df = ratings_with_time.groupBy("user_uid", "rating").agg(
    F.min("watched_time").alias("min_watched_time"),
    F.max("watched_time").alias("max_watched_time")
)

pivot_df = agg_df.groupBy("user_uid").pivot("rating", [8, 9, 10]) \
    .agg(
        F.first("min_watched_time").alias("min"),
        F.first("max_watched_time").alias("max")
    )

for rating in [8, 9, 10]:
    pivot_df = pivot_df \
        .withColumnRenamed(f"{rating}_min", f"feat_min_watched_time_rating_{rating}") \
        .withColumnRenamed(f"{rating}_max", f"feat_max_watched_time_rating_{rating}")

final_df = user_uids_df.join(broadcast(pivot_df), on="user_uid", how="left")

In [22]:
result.explain()
result.show(2, False, True)

== Physical Plan ==
*(7) Project [user_uid#12, feat_min_watched_time_rating_8#145, feat_max_watched_time_rating_8#153, feat_min_watched_time_rating_9#161, feat_max_watched_time_rating_9#169, feat_min_watched_time_rating_10#177, feat_max_watched_time_rating_10#185]
+- *(7) BroadcastHashJoin [user_uid#12], [user_uid#0], LeftOuter, BuildRight, false
   :- *(7) ColumnarToRow
   :  +- FileScan orc hw.user_uids[user_uid#12] Batched: true, DataFilters: [], Format: ORC, Location: InMemoryFileIndex(1 paths)[hdfs://demo/warehouse/tablespace/managed/hive/hw.db/user_uids], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<user_uid:string>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=344]
      +- SortAggregate(key=[user_uid#0], functions=[first(if ((rating#8 <=> 8)) min_watched_time#111 else null, true), first(if ((rating#8 <=> 8)) max_watched_time#113 else null, true), first(if ((rating#8 <=> 9)) min_watched_time#111 else null, true),

[Stage 31:==================================================>       (7 + 1) / 8]

-RECORD 0---------------------------------
 user_uid                        | 110138 
 feat_min_watched_time_rating_8  | 10998  
 feat_max_watched_time_rating_8  | 9435   
 feat_min_watched_time_rating_9  | 10577  
 feat_max_watched_time_rating_9  | 7624   
 feat_min_watched_time_rating_10 | 10331  
 feat_max_watched_time_rating_10 | 9392   
-RECORD 1---------------------------------
 user_uid                        | 412991 
 feat_min_watched_time_rating_8  | 111    
 feat_max_watched_time_rating_8  | 862    
 feat_min_watched_time_rating_9  | 0      
 feat_max_watched_time_rating_9  | 7794   
 feat_min_watched_time_rating_10 | 5959   
 feat_max_watched_time_rating_10 | 7879   
only showing top 2 rows



http://dataops-hadoop-mn-2:18080/history/application_1752758811171_3319/jobs/


Применила broadcast для малых таблиц (user_uids_df, filtered_ratings).
Провела очистку данных на наличие дубликатов. 


In [ ]:
spark.stop()